In [ ]:
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# DATA

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)


# MLP

class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 10)
        )

    def forward(self, x):

        x = x.view(x.size(0), -1)

        return self.network(x)


# CNN

class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.features(x)

        x = x.view(x.size(0), -1)

        x = self.classifier(x)

        return x


# PARAMETER COUNT

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# TRAIN

def train_model(model, epochs=5):

    model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {avg_loss:.4f}"
        )

    training_time = time.time() - start_time

    return training_time


# EVALUATE

def evaluate_model(model):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            total += labels.size(0)

            correct += (
                predictions == labels
            ).sum().item()

    accuracy = 100 * correct / total

    return accuracy


# MAIN

if __name__ == "__main__":

    print("\n")
    print("=" * 60)
    print("TRAINING MLP")
    print("=" * 60)

    mlp = MLP()

    mlp_params = count_parameters(mlp)

    mlp_time = train_model(
        mlp,
        epochs=5
    )

    mlp_accuracy = evaluate_model(
        mlp
    )

    print("\n")
    print("=" * 60)
    print("TRAINING CNN")
    print("=" * 60)

    cnn = CNN()

    cnn_params = count_parameters(cnn)

    cnn_time = train_model(
        cnn,
        epochs=5
    )

    cnn_accuracy = evaluate_model(
        cnn
    )

    print("\n")
    print("=" * 60)
    print("FINAL COMPARISON")
    print("=" * 60)

    print(
        f"{'Model':<10}"
        f"{'Accuracy':<15}"
        f"{'Time(s)':<15}"
        f"{'Parameters':<15}"
    )

    print("-" * 60)

    print(
        f"{'MLP':<10}"
        f"{mlp_accuracy:<15.2f}"
        f"{mlp_time:<15.2f}"
        f"{mlp_params:<15}"
    )

    print(
        f"{'CNN':<10}"
        f"{cnn_accuracy:<15.2f}"
        f"{cnn_time:<15.2f}"
        f"{cnn_params:<15}"
    )